In [1]:
import torch
import pandas as pd
import numpy as np
import random
from pyfaidx import Fasta

import sys
import os
sys.path.append(os.path.abspath("/home1/smaruj/pytorch_akita/"))

# from model import SeqNN
from akita.model import SeqNN

In [ ]:
from utils.data_utils import one_hot_encode_sequence
from utils.optimization_utils import store_tower_output
from utils.data_utils import upper_triangular_to_vector, fragment_indices_in_upper_triangular

In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [5]:
model = SeqNN()
model.load_state_dict(torch.load("/home1/smaruj/pytorch_akita/models/finetuned/mouse/"
    "Hsieh2019_mESC/checkpoints/"
    "Akita_v2_mouse_Hsieh2019_mESC_model0_finetuned.pth", map_location=device))
model.eval()

SeqNN(
  (stochastic_reverse_complement): StochasticReverseComplement()
  (stochastic_shift): StochasticShift()
  (conv_block_1): ConvBlock(
    (conv): Conv1d(4, 128, kernel_size=(15,), stride=(1,), padding=(7,), bias=False)
    (batch_norm): BatchNorm1d(128, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_tower): ConvTower(
    (conv_tower): Sequential(
      (0): ReLU()
      (1): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): ReLU()
      (5): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (6): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (7): MaxPool1d(kernel_size=2, stride=2, paddi

In [6]:
FOLD = 7

In [7]:
df = pd.read_csv(f"/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/genomic_flat_regions/flat_regions_chrom_states_tsv/fold{FOLD}_selected_genomic_windows_centered_chrom_states.tsv", sep="\t")

In [8]:
fasta_file = "/project2/fudenber_735/genomes/mm10/mm10.fa"
genome = Fasta(fasta_file)

In [9]:
bin_size = 2048
cropping_applied = 64
padding_bins = 0
padding = padding_bins * bin_size

slice_0_bins = [256]
slice_0_start = (min(slice_0_bins) + cropping_applied - padding_bins) * bin_size
slice_0_end = (max(slice_0_bins) + 1 + cropping_applied + padding_bins) * bin_size

In [10]:
slice_0_start, slice_0_end

(655360, 657408)

In [11]:
c = -0.5

In [12]:
CTCF_PWM = "/home1/smaruj/IterativeMutagenesis/MA0139.1.meme"

In [13]:
def read_meme_pwm_as_numpy(filename):
    pwm_list = []  # List to store PWM rows
    
    with open(filename, 'r') as file:
        in_matrix_section = False
        
        for line in file:
            line = line.strip()
            
            # Check if we are reading the PWM matrix
            if line.startswith("letter-probability matrix"):
                in_matrix_section = True  # Start reading matrix data
                continue  # Skip this header line
            
            # If we are in the matrix section, process the rows
            if in_matrix_section and line:
                pwm_row = [float(value) for value in line.split()]  # Parse values
                pwm_list.append(pwm_row)  # Append to the PWM list
            
            # If we encounter a new MOTIF or the end of file, stop matrix reading
            if line.startswith("MOTIF") and in_matrix_section:
                break
    
    # Convert the list to a numpy array
    pwm_array = np.array(pwm_list)
    
    return pwm_array

In [14]:
pwm_CTCF = read_meme_pwm_as_numpy(CTCF_PWM)
pwm_CTCF_tensor = torch.from_numpy(pwm_CTCF.T).float()
motifs_dict = {"CTCF": pwm_CTCF_tensor}

In [15]:
from memelite import fimo

In [16]:
ctcf_scan_flank = 15

In [17]:
ctcf_locations_list = []

for row in df.itertuples(index=False):
    chrom, pred_start, pred_end = row.chrom, row.centered_start, row.centered_end
    sequence = genome[row.chrom][pred_start:pred_end]
    
    X = one_hot_encode_sequence(sequence)
    X_tensor = torch.tensor(X)
    
    print(X_tensor.shape)
    
    # modifying X
    X_mod = X_tensor.clone()
    
    mod_slice = torch.load(f"/scratch1/smaruj/generate_genomic_boundary/results/target_-0.5/fold{FOLD}/{chrom}_{pred_start}_{pred_end}_slice.pt", map_location=device)
    
    X_mod[0, :, slice_0_start:slice_0_end] = mod_slice[0, :, :]
        
    slice_to_scan = X_mod[:, :, slice_0_start-ctcf_scan_flank:slice_0_end+ctcf_scan_flank]
    
    hits = fimo(
        motifs=motifs_dict,
        sequences=slice_to_scan.detach().numpy(),
        threshold=1e-4,
        reverse_complement=True
    )[0]
    
    hits["start"] -= ctcf_scan_flank
    hits["end"] -= ctcf_scan_flank
    
    # Store as list of tuples
    locs = [
        (int(s), int(e))
        for s, e in zip(hits["start"], hits["end"])
    ]
    ctcf_locations_list.append(locs)

torch.Size([1, 4, 1310720])
torch.Size([1, 4, 1310720])
torch.Size([1, 4, 1310720])
torch.Size([1, 4, 1310720])
torch.Size([1, 4, 1310720])
torch.Size([1, 4, 1310720])
torch.Size([1, 4, 1310720])
torch.Size([1, 4, 1310720])
torch.Size([1, 4, 1310720])
torch.Size([1, 4, 1310720])
torch.Size([1, 4, 1310720])
torch.Size([1, 4, 1310720])
torch.Size([1, 4, 1310720])
torch.Size([1, 4, 1310720])
torch.Size([1, 4, 1310720])
torch.Size([1, 4, 1310720])
torch.Size([1, 4, 1310720])
torch.Size([1, 4, 1310720])
torch.Size([1, 4, 1310720])
torch.Size([1, 4, 1310720])
torch.Size([1, 4, 1310720])
torch.Size([1, 4, 1310720])
torch.Size([1, 4, 1310720])
torch.Size([1, 4, 1310720])
torch.Size([1, 4, 1310720])
torch.Size([1, 4, 1310720])
torch.Size([1, 4, 1310720])


KeyboardInterrupt: 

In [ ]:
# Add column of lists to your DataFrame
df["ctcf_motif_locs"] = ctcf_locations_list

In [ ]:
df

In [ ]:
df.to_csv(f"/scratch1/smaruj/suppressing_CTCFs/fold{FOLD}_with_positions.tsv", sep="\t", index=False)

In [ ]:
for row in df.itertuples(index=False):
    chrom, pred_start, pred_end = row.chrom, row.centered_start, row.centered_end
    sequence = genome[row.chrom][pred_start:pred_end]
    
    X = one_hot_encode_sequence(sequence)
    X_tensor = torch.tensor(X)
    
    model.eval()
    with torch.no_grad():
        y = model(X_tensor)
    
    torch.save(y, f"/scratch1/smaruj/suppressing_CTCFs/targets/target_{c}/fold{FOLD}/{chrom}_{pred_start}_{pred_end}_target.pt")
    
    # modifying X
    X_mod = X_tensor.clone()
    
    mod_slice = torch.load(f"/scratch1/smaruj/generate_genomic_boundary/overwritten_results/target_{c}/fold{FOLD}/{chrom}_{pred_start}_{pred_end}_slice.pt", map_location=device)
    
    X_mod[0, :, slice_0_start:slice_0_end] = mod_slice[0, :, :]
        
    torch.save(X_mod, f"/scratch1/smaruj/suppressing_CTCFs/ohe_X/fold{FOLD}/{chrom}_{pred_start}_{pred_end}_X.pt")

    store_tower_output(X_mod, model, f"/scratch1/smaruj/suppressing_CTCFs/tower_outputs/fold{FOLD}/{chrom}_{pred_start}_{pred_end}_tower_out.pt")